# Лабораторная работа №2
### Выполнили студенты группы ИНБс-5301-01-00: Симахин С.А., Игнатович В.О.
### Вариант работы: 26



Задание по варианту:
"Вариант 26. Passive-Aggressive. Обучите
sklearn.linear_model.PassiveAggressiveClassifier(random_state=2) . Как этот онлайн-алгоритм
ведёт себя на нашем наборе по сравнению с базовым пайплайном?"


In [1]:
# --- Google Colab, вариант А: архив лежит в корне Google Диска ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
#!unzip -q "https://drive.google.com/file/d/1463kbM03oDOlWuSTNBNfa-st8vWacUxv/" -P 1 -d /content/
!unzip -P 1 -q "/content/drive/MyDrive/lab2_datasets_with_pass.zip" -d /content/

Mounted at /content/drive


In [2]:
import os
import email
from email import policy
import pandas as pd

DATA_DIR = 'datasets/trec07delay/data/'
LABELS_FILE = 'datasets/trec07delay/delay/index'

assert os.path.isdir(DATA_DIR) and os.path.isfile(LABELS_FILE), \
    'Датасет не найден — распакуйте datasets.zip рядом с блокнотом (шаг 0)'

def text_email(path):
    """Извлекает тему и текстовое (text/plain) тело письма — та же функция, что в части 1."""
    try:
        with open(path, 'rb') as f:
            msg = email.message_from_binary_file(f, policy=policy.default)
        subject = str(msg.get('Subject') or '')
        parts = []
        for part in msg.walk():
            if part.get_content_type() == 'text/plain':
                try:
                    content = part.get_content()
                    if isinstance(content, str):
                        parts.append(content)
                except Exception:
                    pass
        return subject + ' ' + ' '.join(parts)
    except Exception:
        return ''

# Читаем метки: имя файла → 'spam'/'ham'
labels = {}
with open(LABELS_FILE) as f:
    for line in f:
        parts = line.split()
        if len(parts) == 2:
            labels[parts[1].split('/')[-1]] = parts[0]

print('Меток прочитано:', len(labels))

Меток прочитано: 10000


In [3]:
# Читаем все письма в список текстов X и список меток y (займёт до минуты)
X, y = [], []
empty = 0
filelist = sorted(labels, key=lambda name: int(name.split('.')[1]))
for filename in filelist:
    text = text_email(DATA_DIR + filename)
    if len(text.split()) <= 2:      # отсев пустых и HTML-only писем по сырому тексту;
        empty += 1                  # в части 1 он строже (после удаления стоп-слов), поэтому числа отличаются
        continue
    X.append(text)
    y.append(labels[filename])

print(f'Прочитано писем: {len(X)}, пропущено пустых: {empty}')
print('Пример текста (первые 200 символов):')
print(X[0][:200])

Прочитано писем: 9868, пропущено пустых: 132
Пример текста (первые 200 символов):
Generic Cialis, branded quality@  


## Шаг 1. Мешок слов: превращаем текст в числа

Модели машинного обучения работают с числами, а не с текстом. Самое простое числовое представление текста — **«мешок слов» (bag-of-words)**:

1. Проходим по всем документам и собираем **словарь** — список всех встретившихся слов.
2. Каждый документ представляем вектором длиной со словарь: на i-м месте стоит **число вхождений** i-го слова в этот документ.

Порядок слов и грамматика при этом теряются — остаётся только «какие слова и сколько раз встретились». Для классификации спама этого, как ни странно, почти достаточно. Разберём на простом примере из трёх «писем»:

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

toy = ['hello mr bear',
       'hello hello gunter',
       'goodbye mr gunter']

toy_vectorizer = CountVectorizer()
toy_counts = toy_vectorizer.fit_transform(toy)

print('Словарь:', list(toy_vectorizer.get_feature_names_out()))
print(toy_counts.toarray())   # строки — письма, столбцы — слова словаря

Словарь: ['bear', 'goodbye', 'gunter', 'hello', 'mr']
[[1 0 0 1 1]
 [0 0 1 2 0]
 [0 1 1 0 1]]


Столбцы идут в алфавитном порядке словаря (`bear`, `goodbye`, `gunter`, `hello`, `mr`), поэтому первая строка `[1 0 0 1 1]` читается так: в письме A по одному разу встречаются `bear`, `hello` и `mr`, а `goodbye` и `gunter` — ни разу.

В реальном корпусе словарь содержит сотни тысяч слов, а в каждом отдельном письме встречается лишь малая их часть — матрица почти целиком нулевая. Поэтому `CountVectorizer` хранит её в **разреженном (sparse)** формате: в памяти лежат только ненулевые элементы.

In [5]:
# Случайное разбиение 70/30; stratify=y сохраняет долю спама одинаковой в обеих частях
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=2, stratify=y)

print(f'train: {len(X_train)} писем, test: {len(X_test)} писем')
print('Доля спама: train {:.1%}, test {:.1%}'.format(
    sum(v == 'spam' for v in y_train) / len(y_train),
    sum(v == 'spam' for v in y_test) / len(y_test)))

train: 6907 писем, test: 2961 писем
Доля спама: train 80.3%, test 80.3%


In [6]:
# Строим словарь ПО ТРЕНИРОВОЧНОЙ ВЫБОРКЕ и векторизуем обе части
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)   # fit: составить словарь + посчитать слова
X_test_counts = vectorizer.transform(X_test)         # только посчитать — словарь уже зафиксирован

print('Размер словаря:', len(vectorizer.vocabulary_))
print('Матрица train:', X_train_counts.shape, '| ненулевых элементов:', X_train_counts.nnz)
print('Матрица test: ', X_test_counts.shape)

Размер словаря: 45547
Матрица train: (6907, 45547) | ненулевых элементов: 462165
Матрица test:  (2961, 45547)


> **Важно:** `fit_transform` вызывается только на train — словарь строится по тренировочным данным. Тестовая выборка проходит только через `transform`: слова, которых не было в train, просто игнорируются. Иначе информация о тесте «протечёт» в обучение.

## Шаг 2. Наивный байесовский классификатор

Идея: для каждого класса (спам / ham) по тренировочной выборке оценивается, с какой вероятностью в нём встречается каждое слово словаря. Для нового письма по **формуле Байеса** вычисляется, к какому классу оно вероятнее относится, — по совокупности его слов.

Классификатор называется «наивным», потому что слова считаются независимыми друг от друга (в реальном языке это, конечно, не так). На практике для текстов метод работает на удивление хорошо, а обучается за доли секунды.

`MultinomialNB` — вариант наивного Байеса для счётчиков слов (наш случай). Чтобы слово, не встречавшееся в train, не обнулило всё произведение вероятностей, применяется **сглаживание Лапласа** (параметр `alpha`, по умолчанию 1.0). Подробная теория — в Приложении А методички.

In [8]:
from sklearn.linear_model import PassiveAggressiveClassifier

clf = PassiveAggressiveClassifier(random_state=2)

clf.fit(X_train_counts, y_train)       # обучение: настройка весов на основе ошибок
y_pred = clf.predict(X_test_counts)    # прогноз для тестовой выборки

print('Первые 10 прогнозов:      ', [str(v) for v in y_pred[:10]])
print('Первые 10 истинных меток: ', y_test[:10])

Первые 10 прогнозов:       ['spam', 'ham', 'spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'spam']
Первые 10 истинных меток:  ['spam', 'ham', 'spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'spam']


In [9]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f'Accuracy: {accuracy_score(y_test, y_pred):.1%}\n')
print(classification_report(y_test, y_pred, digits=3))

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=['spam', 'ham']),
    index=['на самом деле спам', 'на самом деле ham'],
    columns=['прогноз: спам', 'прогноз: ham'])
cm

Accuracy: 98.8%

              precision    recall  f1-score   support

         ham      0.979     0.959     0.969       584
        spam      0.990     0.995     0.992      2377

    accuracy                          0.988      2961
   macro avg      0.984     0.977     0.981      2961
weighted avg      0.988     0.988     0.988      2961



,прогноз: спам,прогноз: ham
на самом деле спам,2365,12
на самом деле ham,24,560


Напоминание по столбцам отчёта `classification_report`:

- **precision** — какая доля писем, названных классом, действительно принадлежит ему;
- **recall** — какая доля писем класса была найдена;
- **f1-score** — гармоническое среднее precision и recall;
- **support** — сколько писем этого класса было в тесте.

## Шаг 3. Сравниваем два подхода

Подставьте в таблицу значения из своих запусков (часть 1 и часть 2):

| Подход | Accuracy | Precision (спам) | Recall (спам) | Чем примечателен |
|---|---|---|---|---|
| Чёрный список (часть 1) | 50.3 % | 85.9 % | 49.1 % | Прозрачные правила, но «мёртвый» список слов |
| Наивный Байес (часть 2) | 97.9 % | 99.4 % | 98.0 % | Обучается на данных сам, использует все слова |
| Пассивно-агрессивный классификатор | 98.8 % | 99.0 % | 99,5 % | Обучается на каждом элементе, динамически меняет правила |

**Выводы**
1. Метод пассивно-агрессивного классификатора позволяет получить наиболее высокий результат по общей доле верных (accuracy) и по доле спам-писем (recall), при этом чистота по спаму (precision по спаму) оказалось немного ниже, чем при методе "наивном Байес".
2. В сравнение с базовым пайплайном результаты оказались примерно такими же, однако данный метод (Passive-Aggressive) является более гибким и расширяемым, а также позволяет обрабатывать поточные данные, например, при онлайн-обучении.